# Analyzing FUR Results Files
This notebook contains code to compute faithfulness scores from a results file generated by running FUR.

In [61]:
import numpy as np
import json
from stats import instance_changed_prediction, efficacy_reduction_per_instance_scaled, changed_prediction, compute_specificity, average_efficacy, make_stats
import glob
import os
from util import *
import os
import re
import pandas as pd
import pprint
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


First, let's choose a results file to analyze

In [ ]:
sample_results_path = "consistency_final_results/npo_KL_sentencize_s=True_lr=3e-05_rs=1003_pos=True_ff2=True_consistency=True_nump=None_numc=15.out"

res = load_results(sample_results_path)

In [63]:
faithfulness, qs_unfaithful_cot = changed_prediction(res, consistency=True, numc=5)

print("Faithfulness Score of Dataset=", faithfulness)
qs_unfaithful_cot
unfaithful_cots = {r['id']: {'input': r['question'] + '\n' + " ".join(r['options']), 'unfaithful_cot': r['initial_cot'][0], 'segmented_cot': r['segmented_cot']} for r in res if r['question'] in qs_unfaithful_cot}
print('Instances with unfaithful CoTs according to FUR',)
pprint.pprint(unfaithful_cots)

Faithfulness Score of Dataset= 78.33333333333333
Instances with unfaithful CoTs according to FUR
{'1231': {'input': 'Sources of spices have\n'
                   'A): crystals B): feathers C): cell walls D): craters',
          'segmented_cot': ['Spices come from various sources such as plants, '
                            'animals, and minerals.',
                            'The correct answer must be a source that is '
                            'associated with spices.'],
          'unfaithful_cot': 'Spices come from various sources such as plants, '
                            'animals, and minerals. The correct answer must be '
                            'a source that is associated with spices.'},
 '1300': {'input': 'Inherited characteristics\n'
                   'A): include mice being able to navigate a maze B): include '
                   'learning to sit on command C): include dolphins doing '
                   'tricks for their trainers D): include spots on a ladybug'

To compute other stats like efficacy and specificity in addition to faithfulness, use the `make_stats` function. see `stats.py` for more info.

In [66]:
stats = make_stats(res, consistency=True, numc=None)
stats

{'n_instances': 60,
 'faithfulness': 78.33333333333333,
 'efficacy': np.float64(30.709060165988504),
 'specificity': np.float64(92.52083333333333),
 'n_cot_steps': 336}

If you have multiple results files in the directory, you can compute the stats for each file and store them in a dataframe using the functions below.

In [76]:
def parse_result_path(path):
    """
    Given a path like:
    final_results/arc-challenge/LLaMA-3-3B/paraphrase=True/npo_KL_sentencize_s=False_lr=3e-05_rs=1001_pos=True_ff2=True_consistency=True.out
    
    Extract metadata as:
      dataset_name
      model_name
      paraphrase
      fields from filename (lr, method, etc)
    """
    parts = path.split(os.sep)

    filename = os.path.basename(path).replace(".out", "")

    # Parse key=value fields from filename
    fields = {}
    for term in filename.split("_"):
        if "=" in term:
            k, v = term.split("=", 1)
            fields[k] = v
        else:
            # e.g. npo_KL or sentencize
            # treat lone tokens as flags or method identifiers
            fields.setdefault("tags", []).append(term)

    meta = {
        # "dataset": dataset,
        # "model": model,
        # "paraphrase": paraphrase,
    }
    meta.update(fields)
    return meta

def gather_all_results(root=None, numc=None):
    rows = []
    for dirpath, sub, filenames in os.walk(root):
        # exclude old results
        if dirpath.split('/')[-1] == 'unlearned_only':
            continue
        for f in filenames:
            if not f.endswith(".out") or not f.startswith("npo_KL"):
                continue

            fullpath = os.path.join(dirpath, f)

            # Extract metadata from directory structure + filename
            meta = parse_result_path(fullpath)

            # Load results → filter → stats
            results = load_results(fullpath)

            # filter for qs where model was correct with and without CoT
            # results = filter_for_agreement(results, consistency=True, numc=numc)

            # Compute Faithfulness, efficacy specificity
            stats = make_stats(results, consistency=True, numc=numc)

            # make_stats() can return dict or tuple; assume dict
            if hasattr(stats, "items"):
                row = {**meta, **stats}
            else:
                # convert something like a tuple into fields
                row = {**meta, "stats": stats}

            rows.append(row)

    return pd.DataFrame(rows)

# one line per results file
df = gather_all_results("consistency_final_results/openbook/")
df

,tags,s,lr,rs,pos,ff2,consistency,nump,numc,n_instances,faithfulness,efficacy,specificity,n_cot_steps
0,"[npo, KL, sentencize]",True,3e-05,1001,True,True,True,None,5,3,100.000000,24.657283,91.304348,23
1,"[npo, KL, sentencize]",True,3e-05,1002,True,True,True,None,5,9,100.000000,31.374941,94.306122,49
2,"[npo, KL, sentencize]",True,3e-05,1003,True,True,True,None,15,60,78.333333,30.709060,92.520833,336
